# T70 — Multi-study global mean surface temperature comparison

**Cluster J: Paleoclimate.**

Global mean surface temperature (GMST) through the Meso-Cenozoic is the single most quoted paleoclimate number — but different modelling groups get different curves because they use different plate reconstructions, different atmospheric CO₂ histories, different model physics, and different ways of taking a *mean*. This notebook plots three published GMST curves — **Farnsworth et al. 2019**, **Landwehrs et al. 2021**, **Li et al. 2023** — alongside Leonard 2025's three PLASIM-GENIE reference-frame estimates, all on the same axes.

The message is not "one curve is right and the others are wrong". It's that **published GMST curves for the same age can differ by 5-10 °C**, and that reference-frame choice is a first-order (not second-order) part of that spread.

## What this notebook produces

1. **§3 — Compute Leonard 2025 GMST per frame.** Area-weighted global mean of `puma_temperature_surface_air` at each age for each of three frames (Merdith 2021 paleomag, Müller 2016 mantle, Torsvik 2019 paleomag).
2. **§4 — Overlay published curves.** Farnsworth 2019 (paleomag frame, HadCM3), Landwehrs 2021 (paleomag frame, PLASIM), Li 2023 (mantle frame, CESM). All on the same axes.
3. **§5 — Cenozoic zoom.** 0-65 Ma detail comparison — the most-studied window and where model-model differences are cleanest to interpret.

## Learning objectives

- Compute an area-weighted global mean from a 2-D field with a `cos(lat)` weighting.
- Combine multiple external GMST time series with heterogeneous ages and CO₂ assumptions on one plot.
- Read a plot showing reference-frame spread as a source of GMST uncertainty.

## Prerequisites and runtime

- Bundled data: `data/leonard_2025_paleoclimate/` (T61 bundle, reused) + `data/paleotemperatures/` (three published GMST CSVs pulled from Leonard 2025's supplementary archive).
- Python: `pandas`, `numpy`, `xarray`, `matplotlib`.
- Runtime: ~15 s (no pyGMT globes — pure time-series plotting).


## Environment + imports


In [ ]:
from pathlib import Path
import os, sys
if Path("../data").exists() and not Path("data").exists():
    os.chdir("..")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

print("Environment")
print(f"  python      {sys.version.split()[0]}")
for _m in (np, pd, xr):
    print(f"  {_m.__name__:11s} {getattr(_m, '__version__', 'n/a')}")


In [ ]:
# === USER CONFIGURATION =====================================================
# Leonard 2025 frame bundle
LEONARD_FRAMES = {
    "Merdith 2021 (paleomag)":   {"dir": Path("data/leonard_2025_paleoclimate/merdith2021_paleomag"),
                                   "colour": "#c0392b", "marker": "o"},
    "Müller 2016 (mantle)":      {"dir": Path("data/leonard_2025_paleoclimate/muller2016_mantle"),
                                   "colour": "#2980b9", "marker": "s"},
    "Torsvik 2019 (paleomag)":   {"dir": Path("data/leonard_2025_paleoclimate/torsvik2019_paleomag"),
                                   "colour": "#8e44ad", "marker": "^"},
}
LEONARD_VAR = "puma_temperature_surface_air"
ALL_AGES_MA = list(range(0, 260, 10))

# Published curves
FARNSWORTH_CSV  = Path("data/paleotemperatures/FarnsworthEtAl2019_GMST.csv")
LANDWEHRS_CSV   = Path("data/paleotemperatures/LandwehrsEtAl2021_GMST.csv")
LI_CSV          = Path("data/paleotemperatures/LiEtAl2023_GMST.csv")

# Cenozoic zoom
CENOZOIC_MAX_MA = 65
# ============================================================================
print(f"  Leonard frames: {list(LEONARD_FRAMES.keys())}")
print(f"  Published curves: Farnsworth 2019, Landwehrs 2021, Li 2023")


## 1. Compute Leonard 2025 area-weighted GMST per frame


In [ ]:
def compute_gmst(frame_dir, ages, var=LEONARD_VAR):
    """Area-weighted GMST time series from Leonard's per-age NCs."""
    records = []
    for age in ages:
        f = frame_dir / f"{age:03d}Ma.nc"
        if not f.exists(): continue
        ds = xr.open_dataset(f)
        if var not in ds: continue
        da = ds[var]
        w = np.cos(np.deg2rad(da.latitude))
        w = w / w.sum()
        gmst = float((da * w).sum(dim=("latitude", "longitude")) / da.sizes["longitude"])
        records.append({"age_ma": age, "gmst_C": gmst})
    return pd.DataFrame(records)

leonard_gmst = {name: compute_gmst(cfg["dir"], ALL_AGES_MA) for name, cfg in LEONARD_FRAMES.items()}
for name, df in leonard_gmst.items():
    print(f"  {name}: GMST range [{df['gmst_C'].min():.1f}, {df['gmst_C'].max():.1f}] °C, "
          f"mean {df['gmst_C'].mean():.1f} °C")


## 2. Load published curves


In [ ]:
# Farnsworth 2019 — has two columns for two CO2 scenarios (1120ppm + 560ppm)
farnsworth = pd.read_csv(FARNSWORTH_CSV, sep=r"\s+")
print(f"  Farnsworth 2019: {len(farnsworth)} rows, columns {list(farnsworth.columns)}")

# Landwehrs 2021 — ensemble of runs; pick the CO2 pathway that matches proxies best
landwehrs = pd.read_csv(LANDWEHRS_CSV)
# Filter to the "proxy" pCO2 pathway for a single canonical curve
lan_proxy = landwehrs[landwehrs["pCO2_pathway"] == "proxy"].copy()
lan_curve = lan_proxy.groupby("Age", as_index=False)["Tann"].mean()
print(f"  Landwehrs 2021: {len(landwehrs)} total rows, {len(lan_curve)} ages after proxy-pathway filter")

# Li 2023 — clean two-column age, gmst
li = pd.read_csv(LI_CSV)
li.columns = [c.strip("\ufeff") for c in li.columns]  # strip BOM
print(f"  Li 2023: {len(li)} rows, columns {list(li.columns)}")


## 3. Overlay plot — full 0-250 Ma


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))

# Leonard 2025 frames
for name, cfg in LEONARD_FRAMES.items():
    df = leonard_gmst[name]
    ax.plot(df["age_ma"], df["gmst_C"], marker=cfg["marker"], lw=1.6, ms=6,
            color=cfg["colour"], alpha=0.9, label=f"Leonard 2025 — {name}")

# Farnsworth 2019 (two CO2 scenarios)
ax.plot(farnsworth["age"], farnsworth["1120ppm"], "d-", lw=1.5, ms=5,
        color="#16a085", alpha=0.8, label="Farnsworth 2019 (HadCM3, 1120 ppm CO2)")
ax.plot(farnsworth["age"], farnsworth["560ppm"], "d--", lw=1.5, ms=5,
        color="#16a085", alpha=0.6, label="Farnsworth 2019 (HadCM3, 560 ppm CO2)")

# Landwehrs 2021
ax.plot(lan_curve["Age"], lan_curve["Tann"], "v-", lw=1.5, ms=5,
        color="#e67e22", alpha=0.8, label="Landwehrs 2021 (PLASIM, proxy-CO2 ensemble mean)")

# Li 2023
ax.plot(li["Age"], li["gmst"], "*-", lw=1.5, ms=7,
        color="#7f8c8d", alpha=0.9, label="Li 2023 (CESM)")

# Reference lines
ax.axhline(14.5, color="black", ls=":", lw=0.8, alpha=0.5)
ax.text(3, 14.5, "  modern (14.5 °C)", va="bottom", ha="left", fontsize=8, color="gray")

ax.set_xlabel("Age (Ma)")
ax.set_ylabel("Global mean surface temperature (°C)")
ax.set_title("Multi-study GMST comparison — Leonard 2025 (3 frames) + Farnsworth 2019 + "
             "Landwehrs 2021 + Li 2023")
ax.set_xlim(0, 260)
ax.set_ylim(10, 34)
ax.invert_xaxis()
ax.legend(loc="upper left", fontsize=8, ncol=1, framealpha=0.9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### How to read the multi-study overlay

- **Leonard 2025 three frames** (red circles, blue squares, purple triangles) — same underlying PLASIM-GENIE simulation, three different absolute plate reference frames. The gap between the highest and lowest at any age is the paper's *reference-frame uncertainty*.
- **Farnsworth 2019** (green diamonds, HadCM3) — two CO₂ scenarios (1120 ppm dashed, 560 ppm solid). Provides an alternative model + CO₂ envelope.
- **Landwehrs 2021** (orange downtriangles, PLASIM) — same model family as Leonard but different plate reconstruction and CO₂ prescription.
- **Li 2023** (grey stars, CESM) — different model + Mesozoic-Cenozoic reconstruction.

**Watch for**

- **Mesozoic warmth**: all curves agree Mesozoic was warmer than modern. Absolute magnitude varies by ~5-10 °C depending on model + CO₂ + frame.
- **Cenozoic cooling**: robust across all studies. Details of PETM (~55 Ma) hot event and Oligocene cooling differ.
- **The Leonard 2025 red-vs-blue gap** (paleomag Merdith vs mantle Müller 2016) is comparable in size to the *between-study* gap (Farnsworth vs Li), suggesting frame choice contributes as much uncertainty as the choice of climate model.


## 4. Cenozoic (0-65 Ma) zoom


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))

for name, cfg in LEONARD_FRAMES.items():
    df = leonard_gmst[name]
    df_c = df[df["age_ma"] <= CENOZOIC_MAX_MA]
    ax.plot(df_c["age_ma"], df_c["gmst_C"], marker=cfg["marker"], lw=2.0, ms=7,
            color=cfg["colour"], alpha=0.9, label=f"Leonard 2025 — {name}")

far_c = farnsworth[farnsworth["age"] <= CENOZOIC_MAX_MA]
ax.plot(far_c["age"], far_c["1120ppm"], "d-", lw=1.6, ms=6,
        color="#16a085", alpha=0.8, label="Farnsworth 2019 (HadCM3, 1120 ppm CO2)")

lan_c = lan_curve[lan_curve["Age"] <= CENOZOIC_MAX_MA]
if len(lan_c) > 0:
    ax.plot(lan_c["Age"], lan_c["Tann"], "v-", lw=1.6, ms=6,
            color="#e67e22", alpha=0.8, label="Landwehrs 2021 (PLASIM)")

li_c = li[li["Age"] <= CENOZOIC_MAX_MA]
if len(li_c) > 0:
    ax.plot(li_c["Age"], li_c["gmst"], "*-", lw=1.6, ms=8,
            color="#7f8c8d", alpha=0.9, label="Li 2023 (CESM)")

# Annotate PETM
ax.axvspan(55.5, 56.5, alpha=0.15, color="red", label="PETM (~56 Ma)")
# Eocene-Oligocene boundary
ax.axvline(33.9, color="blue", ls="--", lw=0.8, alpha=0.5, label="Eocene-Oligocene boundary")

ax.axhline(14.5, color="black", ls=":", lw=0.8, alpha=0.5)
ax.text(1, 14.5, "  modern (14.5 °C)", va="bottom", ha="left", fontsize=8, color="gray")

ax.set_xlabel("Age (Ma)")
ax.set_ylabel("Global mean surface temperature (°C)")
ax.set_title(f"Cenozoic GMST zoom (0-{CENOZOIC_MAX_MA} Ma)  —  "
             "PETM + Eocene-Oligocene boundary highlighted")
ax.set_xlim(0, CENOZOIC_MAX_MA)
ax.invert_xaxis()
ax.legend(loc="upper left", fontsize=8, framealpha=0.9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Headline
print("\n  Cenozoic GMST spread (excluding published curves; frame differences only):")
for age in [0, 20, 40, 60]:
    vals = [leonard_gmst[n][leonard_gmst[n]["age_ma"] == age]["gmst_C"].iloc[0]
            for n in LEONARD_FRAMES if age in leonard_gmst[n]["age_ma"].values]
    if len(vals) >= 2:
        print(f"    {age:>3d} Ma: min {min(vals):.1f}, max {max(vals):.1f}, spread {max(vals)-min(vals):.1f} °C")


### How to read the Cenozoic zoom

- **Eocene-Oligocene boundary (33.9 Ma)** — the sharp cooling step is visible in all curves as a downward inflection around 33-30 Ma.
- **PETM highlight (~56 Ma)** — Leonard 2025 and Landwehrs 2021 are 10-Myr resolution and won't resolve the PETM's brief (~200 kyr) hot excursion; Li 2023 and Farnsworth are similarly averaged.
- **Present-day (0 Ma)** — every study is calibrated approximately to ~14.5 °C modern GMST. Small offsets there indicate model bias.
- **Reference-frame spread** — Leonard's three frames should collapse near-together at 0 Ma (paleomag and mantle frames agree today) and fan out toward the mid-Mesozoic.


## Extend this

- **Add IPCC reference values** — modern GMST 14.5 ± 0.3 °C for calibration.
- **Break down the Landwehrs ensemble.** The bundled CSV has 200+ ensemble members with different CO₂, solar, and land-fraction combinations. Group by CO₂ level or age to see the full model's ECS envelope.
- **Compare Farnsworth CO₂ sensitivity.** Farnsworth 2019 gives *two* CO₂ scenarios per age — the gap between them is the model's climate sensitivity at that age. Add a subplot showing sensitivity vs age.
- **Add proxy GMST estimates.** Judd et al. (2024, *Science*) publish a proxy-based Phanerozoic GMST reconstruction — layering that on top would show model-vs-proxy agreement.
- **Split by ocean vs land.** Leonard 2025's `puma_temperature_surface` gives land-only temperature (via the landsea mask). Land-only GMST is more relevant for continental proxies.

## Related notebooks

- **T61** — Reference-frame uncertainty in reconstructed paleoclimate (map view).
- **T63** — TPW decomposition (why the frames differ).
- **T69** — Ocean gateways through frames (paleogeography view of the same story).

## Sources

- Leonard, J.S., Mather, B.R., Merdith, A.S., Zahirovic, S., Williams, S.E., Müller, R.D. (2025). Polar wander leads to large differences in past climate. *Communications Earth & Environment*.
- **Farnsworth, A., Lunt, D.J., O'Brien, C.L., Foster, G.L., Inglis, G.N., Markwick, P., Pancost, R.D. & Robinson, S.A. (2019).** Climate Sensitivity on Geological Timescales Controlled by Nonlinear Feedbacks and Ocean Circulation. *Geophysical Research Letters* 46, 9880-9889. doi:10.1029/2019GL083574.
- **Landwehrs, J., Feulner, G., Petri, S., Sames, B. & Wagreich, M. (2021).** Investigating Mesozoic Climate Trends and Sensitivities With a Large Ensemble of Climate Model Simulations. *Paleoceanography and Paleoclimatology* 36, e2020PA004134. doi:10.1029/2020PA004134.
- **Li, X., Hu, Y., Yang, J., Wei, M., Guo, J., Lan, J., Lin, Q. et al. (2023).** Climate Variations in the Past 250 Million Years and Contributing Factors. *Paleoceanography and Paleoclimatology* 38, e2022PA004503. doi:10.1029/2022PA004503.
